In [1]:
# ════════════════════════════════════════════════════════════════════════════════════════════
# ScoutIA — VIDÉO RÉSUMÉ D'UN MATCH DÉJÀ ANALYSÉ (sans refaire l'analyse, 0 $) — 21/11/2026 (v2 : + journal)
# Tous les buts + toutes les occasions repérées par l'analyse (réponses Gemini déjà payées).
# ENTRÉES : la sortie du run d'analyse (bout_en_bout_<match>.json, gemini_cache_<match>.jsonl) et la vidéo du match.
# Pas de GPU. ≈ 5 à 10 min. Le module export/resume_match.py doit être sur GitHub.
# ════════════════════════════════════════════════════════════════════════════════════════════
MATCH = "warsage"
VIDEO_NOM = "p1-warsage-2026-09-19.mp4"

import glob, json, os, re, subprocess, sys
REPO = "/kaggle/working/ProjectAIMika"
if not os.path.isdir(REPO):
    os.system(f"git clone -q --depth 1 https://github.com/mikarocco-droid/ProjectAIMika.git {REPO}")
sys.path.insert(0, REPO)
from export.resume_match import monter_resume_match

_un = lambda m: next(iter(sorted(glob.glob(f"/kaggle/input/**/{m}", recursive=True))), None)
be, gc, video = _un(f"bout_en_bout_{MATCH}.json"), _un(f"gemini_cache_{MATCH}*.jsonl"), _un(VIDEO_NOM)
for n, f in (("relais", be), ("réponses Gemini", gc), ("vidéo", video)):
    if not f:
        raise SystemExit(f"❌ {n} introuvable dans les entrées")
buts = json.load(open(be, encoding="utf-8"))["buts"]
occ, vus = [], set()
for l in open(gc, encoding="utf-8"):
    try:
        d = json.loads(l)
    except ValueError:
        continue
    c = d.get("contexte") or {}
    if c.get("fonction") != "classify_highlight_window" or os.path.basename(str(c.get("video_path", ""))) != os.path.basename(video):
        continue
    t = d.get("text") or ""
    o = re.search(r"OUTCOME:\s*\**\s*([A-Z_]+)", t); a = re.search(r"ACTION_TIME:\s*\**\s*(\d+):(\d\d)", t)
    if not (o and a) or o.group(1) not in ("SAVE", "WIDE", "POST", "BLOCKED", "OCCASION"):
        continue
    ta = int(a.group(1)) * 60 + int(a.group(2))
    if any(abs(ta - v) <= 10 for v in vus) or any(abs(ta - float(b["time_abs"])) <= 15 for b in buts):
        continue
    vus.add(ta); occ.append({"time_abs": float(ta), "label": {"SAVE": "arrêt", "WIDE": "tir à côté", "POST": "poteau",
                                                             "BLOCKED": "tir contré"}.get(o.group(1), "occasion")})
# 21/11 : + les tentatives du JOURNAL (descriptions de la validation des coups d'envoi), comme le pipeline : arrêts,
# tirs à côté ou au-dessus, poteaux (les « tirs contrés » du journal ne sont pas repris)
_n_cl = len(occ)
MOTS = [("poteau", r"\b(?:post(?!-)|crossbar|woodwork)\b"),
        ("arrêt", r"\bsave[sd]?\b|\bsaving\b|\bparr(?:y|ies|ied)\b|tips? (?:it )?over"),
        ("tir à côté", r"went wide|goes wide|wide of|over the bar|misses the target|off[- ]target|rolls wide|goes over|went over|high over|over the goal|over/wide|wide/out")]
NEG = re.compile(r"\bno (corner|shot|save|chance|attempt|penalty)|without (a |any )?(shot|save)|\bno shots?\b", re.I)
for l in open(gc, encoding="utf-8"):
    try:
        d = json.loads(l)
    except ValueError:
        continue
    c = d.get("contexte") or {}
    if c.get("fonction") != "validate_kickoff_context_gemini" or os.path.basename(str(c.get("video_path", ""))) != os.path.basename(video):
        continue
    k = float(c.get("kickoff_time_abs") or 0)
    for ph in re.split(r"(?<=[.;])\s+", (d.get("text") or "").split("EVIDENCE:", 1)[-1]):
        if NEG.search(ph):
            continue
        hh = [int(a) * 60 + int(b) for a, b in re.findall(r"(\d{1,3}):(\d{2})", ph)]
        hh = [h for h in hh if k - 35 <= h <= k + 32]
        tm = max(hh) if hh else k          # 21/11 : plage « 86:40-86:48 » → la tentative est vers la fin
        for ty, mo in MOTS:
            if re.search(mo, ph, re.I):
                if not any(abs(tm - o["time_abs"]) <= 12 for o in occ) and not any(abs(tm - float(b["time_abs"])) <= 15 for b in buts):
                    occ.append({"time_abs": float(tm), "label": ty})
                break
occ.sort(key=lambda o: o["time_abs"])
print(f"{len(buts)} but(s), {len(occ)} occasion(s) (dont {len(occ) - _n_cl} du journal)")
r = monter_resume_match(video, buts, occ, "/kaggle/working", nom=f"resume_{MATCH}.mp4")
print(f"✅ {r}" if r else "❌ résumé non produit")

5 but(s), 25 occasion(s) (dont 8 du journal)
  [RÉSUMÉ MATCH] 28 séquence(s), 12.6 min → /kaggle/working/resume_warsage.mp4
     4:17 → 4:37  tir contré
     7:33 → 7:53  tir contré
     8:46 → 9:47  tir contré
     15:11 → 15:31  tir contré
     27:09 → 27:29  tir contré
     29:13 → 29:33  arrêt
     30:30 → 30:50  tir à côté
     32:49 → 33:09  arrêt
     37:11 → 38:06  but possible
     40:36 → 40:56  arrêt
     41:29 → 42:24  but possible
     42:54 → 43:14  poteau
     67:22 → 67:42  tir à côté
     69:04 → 69:24  arrêt
     70:56 → 71:16  tir contré
     82:07 → 83:02  but possible
     86:36 → 86:56  tir à côté
     88:00 → 88:20  tir à côté
     89:55 → 90:15  tir contré
     95:03 → 95:23  tir à côté
     96:34 → 96:54  tir contré
     98:17 → 98:37  arrêt
     102:35 → 103:12  but
     103:42 → 104:02  arrêt
     104:29 → 104:49  arrêt
     106:46 → 107:06  tir à côté
     108:21 → 108:41  tir à côté
     110:17 → 111:12  but possible
✅ /kaggle/working/resume_warsage.mp4
